# 1. SIS + Gaussian forward model

This notebook builds Project Akira's first lens model from the individual numerical steps. The lens is a singular isothermal sphere (SIS) with Einstein radius $\theta_E$. Its deflection is

$$\boldsymbol\alpha(\boldsymbol\theta)=\theta_E\frac{\boldsymbol\theta-\boldsymbol\theta_0}{|\boldsymbol\theta-\boldsymbol\theta_0|},$$

and the single-plane lens equation is

$$\boldsymbol\beta=\boldsymbol\theta-\boldsymbol\alpha(\boldsymbol\theta).$$

Positions and radii are in arcseconds. Coordinate vectors are ordered `(x, y)` even though image arrays use `[y, x]`.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from akira import CartesianGrid, Gaussian, SIS, ray_trace, render_sis_gaussian

print(f'JAX version: {jax.__version__}')
print(f'JAX devices: {jax.devices()}')
print(f'64-bit enabled: {jax.config.x64_enabled}')

## Define the image grid and physical model

`supersampling=2` evaluates four sub-pixel centres per detector pixel and averages their surface brightness. The Gaussian uses a circularized scale radius and can be elliptical through `q=b/a` and `phi`, measured anticlockwise from positive x.

In [ ]:
grid = CartesianGrid(shape=(160, 160), pixel_scale=0.025, supersampling=2)
lens = SIS(einstein_radius=1.0, centre=(0.0, 0.0))
source = Gaussian(
    intensity=1.0,
    sigma=0.09,
    centre=(0.08, -0.04),
    q=0.72,
    phi=0.35,
)

subpixel_coordinates = grid.subpixel_coordinates()
print('sub-pixel coordinate shape:', subpixel_coordinates.shape)
print('image shape:', grid.shape)

## Execute the forward model one step at a time

The public profile objects validate parameters. Their numerical methods operate on arrays and remain compatible with JAX compilation and differentiation. Lensing conserves surface brightness: the source profile is simply evaluated at the ray-traced source coordinate.

In [ ]:
deflections = lens.deflection(subpixel_coordinates)
source_coordinates = ray_trace(subpixel_coordinates, deflections)
subpixel_brightness = source.brightness(source_coordinates)
manual_image = subpixel_brightness.mean(axis=2)

public_image = render_sis_gaussian(grid, lens, source)
np.testing.assert_allclose(manual_image, public_image, rtol=1e-14, atol=1e-14)
print('Manual equation pipeline exactly matches the public renderer.')

## Visualize the source and lensed image

In [ ]:
pixel_coordinates = grid.coordinates()
unlensed_source = source.brightness(pixel_coordinates)
deflection_magnitude = jnp.linalg.vector_norm(lens.deflection(pixel_coordinates), axis=-1)
extent = [
    float(pixel_coordinates[0, 0, 0]), float(pixel_coordinates[0, -1, 0]),
    float(pixel_coordinates[0, 0, 1]), float(pixel_coordinates[-1, 0, 1]),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, image, title in zip(
    axes,
    [unlensed_source, public_image, deflection_magnitude],
    ['Unlensed Gaussian source', 'SIS-lensed image', 'Deflection magnitude'],
):
    display = ax.imshow(np.asarray(image), origin='lower', extent=extent, cmap='magma')
    ax.set(title=title, xlabel='x [arcsec]', ylabel='y [arcsec]')
    fig.colorbar(display, ax=ax, shrink=0.82)
plt.show()

## Verify the Einstein-ring scale

For a centred compact source, the maximum surface brightness should lie close to the SIS Einstein radius. An even-sized grid avoids sampling the mathematical singularity exactly at the origin.

In [ ]:
centred_source = Gaussian(intensity=1.0, sigma=0.04)
ring = render_sis_gaussian(grid, lens, centred_source)
radii = jnp.linalg.vector_norm(pixel_coordinates, axis=-1)
peak_radius = float(radii.reshape(-1)[jnp.argmax(ring)])

print(f'Expected Einstein radius: {lens.einstein_radius:.6f} arcsec')
print(f'Peak ring radius:         {peak_radius:.6f} arcsec')
assert abs(peak_radius - lens.einstein_radius) < grid.pixel_scale